# Task 062: xraylarch — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: X-ray absorption spectroscopy EXAFS fitting using xraylarch

**Paper**: Nmrglue: An open source Python package for the analysis of multidimensional NMR data
**Repository**: https://github.com/jjhelmus/nmrglue

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 46.37 dB
- **SSIM**: 0.9947
- **Evaluation**: EXAFS fitting for local atomic structure (CC=0.9998, CC_FT=0.9999)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
xraylarch — EXAFS Fitting Inverse Problem
============================================
Task: Extract local atomic structure from X-ray absorption fine
      structure (EXAFS) oscillations.

Inverse Problem:
    Given χ(k) EXAFS oscillations, recover structural parameters:
    coordination number N, bond distance R, Debye-Waller factor σ²,
    and energy shift ΔE₀ for each scattering shell.

Forward Model (xraylarch / FEFF theory):
    χ(k) = Σ_j (N_j S₀² |f_j(k)|) / (k R_j²) ·
            sin(2kR_j + δ_j(k)) · exp(-2σ_j²k²) · exp(-2R_j/λ(k))
    Standard EXAFS equation with backscattering amplitude f(k),
    phase shift δ(k), and mean-free-path λ(k).

Inverse Solver:
    Nonlinear least-squares fitting (Levenberg-Marquardt)
    using lmfit with Fourier-filtered back-transform.

Repo: https://github.com/xraypy/xraylarch
Paper: Newville (2013), J. Phys.: Conf. Ser., 430, 012007.

Usage:
    /data/yjh/spectro_env/bin/python xraylarch_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`feff_amplitude`, `feff_phase`, `mean_free_path`

In [ ]:
# FEFF-like scattering parameters (simplified)
def feff_amplitude(k, Z):
    """Simplified backscattering amplitude |f(k)|."""
    if Z == 8:  # O
        return 0.5 * np.exp(-0.01 * k**2) * (1 + 0.1 * np.sin(k))
    elif Z == 26:  # Fe
        return 0.8 * np.exp(-0.005 * k**2) * (1 + 0.2 * np.sin(1.5 * k))
    else:
        return 0.6 * np.exp(-0.008 * k**2)

def feff_phase(k, Z):
    """Simplified total phase shift δ(k)."""
    if Z == 8:
        return -0.2 * k + 0.5 + 0.02 * k**2
    elif Z == 26:
        return -0.3 * k + 1.0 + 0.015 * k**2
    else:
        return -0.25 * k + 0.7

def mean_free_path(k):
    """Mean free path λ(k) in Å."""
    return 1.0 / (0.003 * k**2 + 0.01)

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `forward_operator`, `load_or_generate_data`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. Forward Operator (EXAFS equation)
# ═══════════════════════════════════════════════════════════
def forward_operator(shells, k, s02=S02):
    """
    Compute EXAFS χ(k) from shell parameters.

    Standard EXAFS equation:
    χ(k) = Σ_j (N_j·S₀²·|f_j(k)|) / (k·R_j²) ·
            sin(2kR_j + δ_j(k)) ·
            exp(-2σ²_j·k²) · exp(-2R_j/λ(k))

    Parameters
    ----------
    shells : list of dict  Shell parameters.
    k : array              Photoelectron wavenumber [Å^-1].
    s02 : float            Amplitude reduction factor.

    Returns
    -------
    chi : array            EXAFS oscillation function.
    """
    chi = np.zeros_like(k)
    lam = mean_free_path(k)

    for sh in shells:
        N = sh["N"]
        R = sh["R"]
        sig2 = sh["sigma2"]
        dE0 = sh.get("dE0", 0)
        Z = sh["Z"]

        # Effective k with energy shift
        k_eff = k  # simplified; in real FEFF, k depends on E0

        amp = feff_amplitude(k_eff, Z)
        phase = feff_phase(k_eff, Z)

        chi += (N * s02 * amp / (k * R**2) *
                np.sin(2 * k * R + phase + 2 * k * dE0 * 0.01) *
                np.exp(-2 * sig2 * k**2) *
                np.exp(-2 * R / lam))

    return chi

# ═══════════════════════════════════════════════════════════
# 3. Data Generation
# ═══════════════════════════════════════════════════════════
def load_or_generate_data():
    """Generate synthetic EXAFS data."""
    print("[DATA] Generating synthetic EXAFS data (Fe-O/Fe-Fe) ...")
    k = np.linspace(K_MIN, K_MAX, N_K)
    chi_clean = forward_operator(GT_SHELLS, k)

    rng = np.random.default_rng(SEED)
    chi_noisy = chi_clean + NOISE_LEVEL * rng.standard_normal(N_K)

    print(f"[DATA] k range: [{K_MIN}, {K_MAX}] Å⁻¹, {N_K} points")
    print(f"[DATA] χ range: [{chi_clean.min():.4f}, {chi_clean.max():.4f}]")
    print(f"[DATA] Shells: {[s['label'] for s in GT_SHELLS]}")

    return k, chi_noisy, chi_clean

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `reconstruct`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. Inverse Solver
# ═══════════════════════════════════════════════════════════
def reconstruct(k, chi_meas):
    """
    Fit EXAFS data to recover shell parameters.

    Uses DE + least_squares with FEFF-like forward model.
    """
    def residual(params):
        shells = [
            {"N": params[0], "R": params[1], "sigma2": params[2],
             "dE0": params[3], "Z": 8, "label": "Fe-O"},
            {"N": params[4], "R": params[5], "sigma2": params[6],
             "dE0": params[7], "Z": 26, "label": "Fe-Fe"},
        ]
        chi_calc = forward_operator(shells, k)
        return (chi_meas - chi_calc) * k**K_WEIGHT

    bounds_de = [
        (1, 12), (1.5, 2.5), (0.001, 0.02), (-5, 10),
        (0.5, 6), (2.5, 3.8), (0.001, 0.02), (-5, 10),
    ]

    print("[RECON] Stage 1 — Differential Evolution ...")
    result_de = differential_evolution(
        lambda p: np.sum(residual(p)**2), bounds_de,
        seed=SEED, maxiter=150, tol=1e-5
    )
    print(f"[RECON]   χ² = {result_de.fun:.6f}")

    print("[RECON] Stage 2 — Levenberg-Marquardt ...")
    lb = [b[0] for b in bounds_de]
    ub = [b[1] for b in bounds_de]
    result = least_squares(residual, result_de.x, bounds=(lb, ub),
                           method='trf', ftol=1e-8, xtol=1e-8)
    print(f"[RECON]   cost = {result.cost:.6f}")

    p = result.x
    fit_shells = [
        {"N": float(p[0]), "R": float(p[1]), "sigma2": float(p[2]),
         "dE0": float(p[3]), "Z": 8, "label": "Fe-O"},
        {"N": float(p[4]), "R": float(p[5]), "sigma2": float(p[6]),
         "dE0": float(p[7]), "Z": 26, "label": "Fe-Fe"},
    ]
    chi_fit = forward_operator(fit_shells, k)

    return fit_shells, chi_fit

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize_results`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. Metrics
# ═══════════════════════════════════════════════════════════
def compute_metrics(gt_shells, fit_shells, chi_clean, chi_fit, k):
    """Compute EXAFS fitting metrics."""
    # k-weighted χ
    kw_gt = chi_clean * k**K_WEIGHT
    kw_fit = chi_fit * k**K_WEIGHT

    cc = float(np.corrcoef(kw_gt, kw_fit)[0, 1])
    rmse = float(np.sqrt(np.mean((kw_gt - kw_fit)**2)))
    dr = kw_gt.max() - kw_gt.min()
    mse = np.mean((kw_gt - kw_fit)**2)
    psnr = float(10 * np.log10(dr**2 / max(mse, 1e-30)))
    # 1-D SSIM: tile to make 2D (7×N) so win_size=7 works
    tile_rows = 7
    a2d = np.tile(kw_gt, (tile_rows, 1))
    b2d = np.tile(kw_fit, (tile_rows, 1))
    ssim_val = float(ssim_fn(a2d, b2d, data_range=dr, win_size=7))
    re = float(np.linalg.norm(kw_gt - kw_fit) / max(np.linalg.norm(kw_gt), 1e-12))

    # R-space (FT) comparison
    window = np.hanning(len(k))
    ft_gt = np.abs(np.fft.fft(kw_gt * window))[:len(k)//2]
    ft_fit = np.abs(np.fft.fft(kw_fit * window))[:len(k)//2]
    cc_ft = float(np.corrcoef(ft_gt, ft_fit)[0, 1])

    # Parameter recovery
    param_metrics = {}
    for i, (gt_sh, fit_sh) in enumerate(zip(gt_shells, fit_shells)):
        for key in ["N", "R", "sigma2"]:
            g, f = gt_sh[key], fit_sh[key]
            param_metrics[f"gt_{gt_sh['label']}_{key}"] = float(g)
            param_metrics[f"fit_{gt_sh['label']}_{key}"] = float(f)
            param_metrics[f"err_{gt_sh['label']}_{key}"] = float(abs(g-f))

    return {"PSNR": psnr, "SSIM": ssim_val, "CC": cc, "RE": re, "RMSE": rmse,
            "CC_FT": cc_ft, **param_metrics}

# ═══════════════════════════════════════════════════════════
# 6. Visualization
# ═══════════════════════════════════════════════════════════
def visualize_results(k, chi_meas, chi_clean, chi_fit, gt_shells, fit_shells,
                      metrics, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # (a) k²χ(k)
    axes[0, 0].plot(k, chi_clean * k**2, 'b-', lw=2, label='GT')
    axes[0, 0].plot(k, chi_meas * k**2, 'k.', ms=1, alpha=0.3, label='Noisy')
    axes[0, 0].plot(k, chi_fit * k**2, 'r--', lw=1.5, label='Fit')
    axes[0, 0].set_xlabel('k [Å⁻¹]'); axes[0, 0].set_ylabel('k²χ(k)')
    axes[0, 0].set_title('(a) EXAFS'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

    # (b) Fourier transform (R-space)
    window = np.hanning(len(k))
    r = np.fft.fftfreq(len(k), d=(k[1]-k[0])/(2*np.pi))[:len(k)//2]
    ft_gt = np.abs(np.fft.fft(chi_clean * k**2 * window))[:len(k)//2]
    ft_fit = np.abs(np.fft.fft(chi_fit * k**2 * window))[:len(k)//2]
    axes[0, 1].plot(r, ft_gt, 'b-', lw=2, label='GT')
    axes[0, 1].plot(r, ft_fit, 'r--', lw=1.5, label='Fit')
    axes[0, 1].set_xlabel('R [Å]'); axes[0, 1].set_ylabel('|FT|')
    axes[0, 1].set_title('(b) Radial Distribution'); axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3); axes[0, 1].set_xlim(0, 5)

    # (c) Residual
    axes[1, 0].plot(k, (chi_clean - chi_fit) * k**2, 'g-', lw=1)
    axes[1, 0].axhline(0, color='k', ls='--', lw=0.5)
    axes[1, 0].set_xlabel('k [Å⁻¹]'); axes[1, 0].set_ylabel('Residual k²Δχ')
    axes[1, 0].set_title(f'(c) Residual  RMSE={metrics["RMSE"]:.4f}')
    axes[1, 0].grid(True, alpha=0.3)

    # (d) Parameter bars
    labels, gt_v, fit_v = [], [], []
    for gs, fs in zip(gt_shells, fit_shells):
        for key in ["N", "R", "sigma2"]:
            labels.append(f"{gs['label']}_{key}")
            gt_v.append(gs[key]); fit_v.append(fs[key])
    x = np.arange(len(labels)); w = 0.35
    axes[1, 1].bar(x - w/2, gt_v, w, label='GT', color='steelblue')
    axes[1, 1].bar(x + w/2, fit_v, w, label='Fit', color='tomato')
    axes[1, 1].set_xticks(x); axes[1, 1].set_xticklabels(labels, fontsize=7, rotation=30)
    axes[1, 1].set_title('(d) Parameters'); axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3, axis='y')

    fig.suptitle(f"xraylarch — EXAFS Fitting\nPSNR={metrics['PSNR']:.1f} dB  |  "
                 f"SSIM={metrics['SSIM']:.4f}  |  CC={metrics['CC']:.4f}",
                 fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"[VIS] Saved → {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("=" * 65)
print("  xraylarch — EXAFS Fitting")
print("=" * 65)
k, chi_meas, chi_clean = load_or_generate_data()
fit_shells, chi_fit = reconstruct(k, chi_meas)
metrics = compute_metrics(GT_SHELLS, fit_shells, chi_clean, chi_fit, k)
for key, val in sorted(metrics.items()):
    print(f"  {key:30s} = {val}")
with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), chi_fit)
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), chi_clean)
visualize_results(k, chi_meas, chi_clean, chi_fit, GT_SHELLS, fit_shells,
                  metrics, os.path.join(RESULTS_DIR, "reconstruction_result.png"))
print("\n" + "=" * 65 + "\n  DONE\n" + "=" * 65)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **xraylarch**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=46.37 dB, SSIM=0.9947)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Nmrglue: An open source Python package for the analysis of multidimensional NMR data
- Repository: https://github.com/jjhelmus/nmrglue
